In [1]:
import time
import importlib
import cv2
import numpy as np
from ugot import ugot
import pose_yolo

# Reload the pose control module so any edits to pose_yolo.py take effect
# without restarting the kernel.
importlib.reload(pose_yolo)
from pose_yolo import run_pose_control_inline

# Utilities for showing live camera frames directly inside the notebook
from IPython.display import display, clear_output, Image
from PIL import Image as Image2

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize('192.168.88.1')  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models([
    'color_recognition',
    'word_recognition',
    'line_recognition',
    'apriltag_qrcode'
])

192.168.88.1:50051


_InactiveRpcError: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "failed to connect to all addresses; last error: UNKNOWN: ipv4:192.168.88.1:50051: tcp handshaker shutdown"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_status:14, grpc_message:"failed to connect to all addresses; last error: UNKNOWN: ipv4:192.168.88.1:50051: tcp handshaker shutdown"}"
>

In [21]:
def AP_centralization_approaching(distance=0.15, gap=20, fwd_spd=10, strafe_spd=10):
    """
    Drive toward a detected AprilTag, keeping it centered in the camera frame.

    Parameters:
        distance  (float): Stop when the tag is within this many meters (default 0.15 m).
        gap       (int):   Pixel tolerance around center (320 px) before strafing (default 20 px).
        fwd_spd   (int):   Forward drive speed percentage (default 10 cm/s).
        strafe_spd(int):   Left/right correction speed percentage (default 10 cm/s).
    """
    # Get an initial reading to confirm a tag is visible before entering the loop.
    AP_info = got.get_apriltag_total_info()
    AP_x = AP_info[0][1] # Horizontal pixel position of the tag (0=left, 640=right)
    AP_distance = AP_info[0][6] # Estimated distance to the tag in meters

    while True:
        # Refresh tag data every iteration for responsive corrections.
        AP_info = got.get_apriltag_total_info()
        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]

        if AP_x < 320 - gap:
            # Tag is to the LEFT of center — strafe left to re-align.
            # mecanum_move_xyz(x, y, z): x=strafe, y=forward, z=rotation
            print("Tag is to the LEFT of center — strafe left to re-align.")
            got.mecanum_move_xyz(-strafe_spd, strafe_spd, 0)
        elif AP_x > 320 + gap:
            # Tag is to the RIGHT of center — strafe right to re-align.
            print("Tag is to the RIGHT of center — strafe right to re-align.")
            got.mecanum_move_xyz(strafe_spd, strafe_spd, 0)
        elif AP_distance > distance:
            # Tag is centered but still too far — drive straight forward.
            print("Tag is centered but still too far — drive straight forward.")
            got.mecanum_move_xyz(0, fwd_spd, 0)
        else:
            # Tag is centered AND within target distance — stop and exit.
            got.mecanum_stop()
            print("It's too close, let's stop.")
            break
def pick_up():
    """
    Pick up the object identified by the closest visible AprilTag.

    Assumes the robot is already aligned and close to the target
    (i.e., AP_centralization_approaching() has just completed).
    """
    # Read the tag's current position and distance for arm targeting.
    AP_info = got.get_apriltag_total_info()
    try:
        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]

        # Move arm to a neutral ready position and open the gripper.
        # joint_control(j1, j2, j3, duration_ms): j2=30, j3=30 tilts arm slightly forward.
        got.mechanical_joint_control(0, 30, 30, 1000)
        got.mechanical_clamp_release() # Open gripper before extending arm
        time.sleep(2) # Wait for gripper to fully open

        # Calculate arm joint angles based on the tag's camera position.
        # joint1 (base): convert pixel offset from center to degrees.
        #   Negative factor corrects for the camera being mirrored horizontally.
        joint1 = int((AP_x - 320) * -1 / 10)

        # joint3 (furthest): convert distance (m) to an extension angle.
        # The -80 offset accounts for the arm's resting angle calibration.
        joint3 = int(AP_distance * 100 - 80)

        # Move arm to the computed pick-up pose.
        got.mechanical_joint_control(joint1, 0, joint3, 500)
        print(f"Joint1 value is: {joint1}, Joint3 value is: {joint3}.")
        time.sleep(1) # Wait for arm to reach the target pose

        # Grasp the object and lift the arm back to the carry position.
        got.mechanical_clamp_close()
        time.sleep(2)  # Wait for gripper to fully close before lifting
        got.mechanical_joint_control(0, 30, 30, 1000)  # Return arm to neutral carry pose
    except IndexError:
        print("AprilTag is too close!")

In [29]:
try:

    while True:
        AP_info = got.get_apriltag_total_info()

        if AP_info:
            print("see tag")
            AP_centralization_approaching(
                distance=0.185, gap=22, fwd_spd=7, strafe_spd=5
            )
            pick_up()
            print("Picked up!")
            time.sleep(2)
            break
        else:
            print("see no tag, moving up")
            got.mecanum_move_speed(0, 15)

    
except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done")
            

see tag
Tag is centered but still too far — drive straight forward.
Tag is centered but still too far — drive straight forward.
Tag is centered but still too far — drive straight forward.
Tag is centered but still too far — drive straight forward.
Tag is centered but still too far — drive straight forward.
Tag is centered but still too far — drive straight forward.
Tag is centered but still too far — drive straight forward.
Tag is centered but still too far — drive straight forward.
Tag is centered but still too far — drive straight forward.
Tag is to the RIGHT of center — strafe right to re-align.
Tag is to the RIGHT of center — strafe right to re-align.
Tag is centered but still too far — drive straight forward.
Tag is centered but still too far — drive straight forward.
Tag is centered but still too far — drive straight forward.
Tag is centered but still too far — drive straight forward.
Tag is centered but still too far — drive straight forward.
Tag is centered but still too far — 